# Análisis de Similitud de Malware — Índice de Jaccard Modificado

**- Juan Pablo Solis**  
**- Andre Yatmian Jo Mai**

---

Se utiliza el **Índice de Jaccard Modificado** para analizar similitud entre muestras de malware, usando dos características:

| Característica | Descripción |
|---|---|
| **API Calls (Llamadas a funciones)** | Conjunto de imports `DLL:función` presentes en la muestra |
| **Strings** | Conjunto de nombres de función únicos (sin la DLL) |

Se analizan **tres umbrales de similitud**: `0.3`, `0.5`, `0.7`.

Para cada característica y umbral se genera:
- Un **grafo por familia** de malware
- Un **grafo de todo el conjunto**

## 1. Imports y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import re
import warnings
warnings.filterwarnings('ignore')

THRESHOLDS = [0.3, 0.5, 0.7]

FAMILY_COLORS = [
    '#1f77b4','#d62728','#ff7f0e','#2ca02c','#9467bd',
    '#8c564b','#e377c2','#17becf','#bcbd22','#7f7f7f'
]

print('OK')

## 2. Carga del Dataset

In [ ]:
df = pd.read_csv('malware_dataset.csv')
print('Shape:', df.shape)
df.head(3)

## 3. Extracción de Familias y Construcción de Conjuntos

- **Familia**: prefijo del nombre de archivo antes del guión bajo (`NBV_...` → `NBV`). Archivos sin prefijo quedan como `Generic`.
- **api_set**: conjunto de columnas `DLL:función` donde el valor es `1`.
- **string_set**: conjunto de nombres de función únicos (parte después de `:`).

In [ ]:
# ── Columnas de API calls (tienen formato 'DLL:función') ───────────────────────
import_cols = [c for c in df.columns if ':' in c]
print(f'Columnas de API calls: {len(import_cols)}')

# ── Extraer familia del nombre de archivo ──────────────────────────────────────
def extract_family(filename):
    """Prefijo alfabético antes del primer guión bajo = familia."""
    m = re.match(r'^([A-Za-z][A-Za-z0-9]{1,7})_[0-9A-Fa-f]', filename)
    if m:
        return m.group(1)
    m2 = re.match(r'^([A-Za-z]{2,5}[0-9]{0,3})[0-9A-F]{6}', filename)
    if m2 and len(m2.group(1)) <= 6:
        return m2.group(1)
    return 'Generic'

df['family'] = df['file'].apply(extract_family)

# ── Construir conjuntos por muestra ───────────────────────────────────────────
def build_api_set(row):
    """Conjunto de API calls presentes (valor == 1)."""
    return frozenset(col for col in import_cols if row[col] == 1)

def build_string_set(api_set):
    """Conjunto de nombres de función (parte después de ':')."""
    return frozenset(api.split(':')[1] for api in api_set)

df['api_set']    = df.apply(build_api_set, axis=1)
df['string_set'] = df['api_set'].apply(build_string_set)

# ── Resumen ────────────────────────────────────────────────────────────────────
print()
print('Muestras por familia:')
print(df['family'].value_counts().to_string())
print()
print('Ejemplo muestra 1:')
print('  Familia :', df.loc[1, 'family'])
print('  api_set :', list(df.loc[1, 'api_set'])[:4], '...')
print('  str_set :', list(df.loc[1, 'string_set'])[:4], '...')

## 4. Índice de Jaccard Modificado

La versión **modificada** usa $\min(|A|, |B|)$ en lugar de $|A \cup B|$ como base del denominador, reduciendo la penalización por diferencias de tamaño entre conjuntos:

$$J_{mod}(A, B) = \frac{|A \cap B|}{\min(|A|, |B|) + \alpha \cdot (|A \cup B| - |A \cap B|)}$$

- $\alpha = 1.0$ → Jaccard estándar  
- $\alpha = 0.5$ → versión modificada (menos penalización por tamaño desigual)

In [ ]:
def jaccard_modificado(set_a: frozenset, set_b: frozenset, alpha: float = 0.5) -> float:
    """
    Índice de Jaccard Modificado.

    Parámetros
    ----------
    set_a, set_b : frozenset — conjuntos a comparar
    alpha        : float en (0, 1] — factor de penalización
                   1.0 = Jaccard estándar | 0.5 = versión modificada

    Retorna
    -------
    float en [0, 1]
    """
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0

    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    mini  = min(len(set_a), len(set_b))

    denom = mini + alpha * (union - inter)
    return inter / denom if denom > 0 else 0.0


# ── Prueba ─────────────────────────────────────────────────────────────────────
A = frozenset({'LoadLibraryA', 'ExitProcess', 'VirtualProtect'})
B = frozenset({'LoadLibraryA', 'ExitProcess', 'GetProcAddress', 'Sleep', 'CreateThread'})

print('Prueba con conjuntos de distinto tamaño:')
print(f'  |A|={len(A)}  |B|={len(B)}')
print(f'  Jaccard estándar   (α=1.0): {jaccard_modificado(A, B, 1.0):.4f}')
print(f'  Jaccard modificado (α=0.5): {jaccard_modificado(A, B, 0.5):.4f}')
print()
print('El modificado puntúa más alto porque penaliza menos la diferencia de tamaño.')

## 5. Funciones Auxiliares de Grafos

In [ ]:
def build_similarity_graph(indices, feature_sets, threshold, alpha=0.5):
    """
    Construye un grafo de similitud NetworkX.

    Nodo  = índice de la muestra en df
    Arista = par con similitud >= threshold
    Peso   = valor de similitud
    """
    G = nx.Graph()
    G.add_nodes_from(indices)

    n = len(indices)
    for i in range(n):
        for j in range(i + 1, n):
            sim = jaccard_modificado(feature_sets[i], feature_sets[j], alpha)
            if sim >= threshold:
                G.add_edge(indices[i], indices[j], weight=sim)
    return G


def draw_graph(G, df_ref, title, node_color, ax,
               color_by_family=False, family_color_map=None):
    """
    Dibuja el grafo en un Axes.

    - Etiquetas: primeros 8 chars del hash del archivo
    - Tamaño de nodo ∝ grado
    - Grosor de arista ∝ peso (similitud)
    """
    ax.set_title(
        f'{title}\n'
        f'Nodos={G.number_of_nodes()}  '
        f'Aristas={G.number_of_edges()}  '
        f'Densidad={nx.density(G):.3f}',
        fontsize=9, pad=6
    )
    ax.axis('off')

    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'Sin conexiones', ha='center', va='center',
                transform=ax.transAxes, color='gray', fontsize=9)
        return

    pos = nx.spring_layout(G, seed=42, k=1.4)

    deg = dict(G.degree())
    sizes = [140 + deg[n] * 90 for n in G.nodes()]

    if color_by_family and family_color_map:
        colors = [family_color_map.get(df_ref.loc[n, 'family'], '#aaaaaa')
                  for n in G.nodes()]
    else:
        colors = node_color

    edges = list(G.edges())
    widths = [G[u][v]['weight'] * 4 for u, v in edges] if edges else []

    nx.draw_networkx_nodes(G, pos, node_size=sizes,
                           node_color=colors, alpha=0.88, ax=ax)
    nx.draw_networkx_edges(G, pos, width=widths,
                           edge_color='#444444', alpha=0.45, ax=ax)

    # Etiquetas: 8 primeros chars del hash (parte después del prefijo de familia)
    labels = {}
    for n in G.nodes():
        fname = df_ref.loc[n, 'file']
        # Si tiene underscore, tomar la parte hash; si no, los primeros 8 chars
        parts = fname.split('_', 1)
        labels[n] = parts[-1][:8] if len(parts) > 1 else fname[:8]

    nx.draw_networkx_labels(G, pos, labels=labels, font_size=6, ax=ax)


print('Funciones de grafos definidas.')

---
# CARACTERÍSTICA 1: Llamadas a Funciones (API Calls)

Cada muestra se representa como el **conjunto de imports** `DLL:función` con valor `1`.

In [ ]:
families       = sorted(df['family'].unique())
n_fam          = len(families)
family_color_map = {f: FAMILY_COLORS[i % len(FAMILY_COLORS)]
                    for i, f in enumerate(families)}

print(f'Familias ({n_fam}): {families}')

## 6a. API Calls — Grafo por Familia

Una figura por umbral. Cada subplot = una familia.

In [ ]:
for threshold in THRESHOLDS:
    # Calcular número de columnas/filas para los subplots
    ncols = min(n_fam, 5)
    nrows = (n_fam + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(5.5 * ncols, 5 * nrows))
    axes_flat = np.array(axes).flatten() if n_fam > 1 else [axes]

    fig.suptitle(
        f'API Calls — Grafo por Familia  |  Umbral = {threshold}  (α = 0.5)',
        fontsize=13, fontweight='bold', y=1.01
    )

    for k, family in enumerate(families):
        sub     = df[df['family'] == family]
        idxs    = list(sub.index)
        sets    = list(sub['api_set'])
        color   = FAMILY_COLORS[k % len(FAMILY_COLORS)]

        G = build_similarity_graph(idxs, sets, threshold, alpha=0.5)
        draw_graph(G, df, title=f'Familia: {family}',
                   node_color=color, ax=axes_flat[k])

    for ax in axes_flat[n_fam:]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'api_calls_familia_t{int(threshold*10)}.png',
                dpi=130, bbox_inches='tight')
    plt.show()
    print(f'  Umbral {threshold} completado.')

## 6b. API Calls — Grafo de Todo el Conjunto

Una figura por umbral. Nodos coloreados por familia.

In [ ]:
for threshold in THRESHOLDS:
    all_idxs = list(df.index)
    all_sets = list(df['api_set'])

    G_all = build_similarity_graph(all_idxs, all_sets, threshold, alpha=0.5)

    fig, ax = plt.subplots(figsize=(14, 10))

    draw_graph(G_all, df,
               title=f'API Calls — Todo el Conjunto  |  Umbral = {threshold}  (α = 0.5)',
               node_color=None, ax=ax,
               color_by_family=True, family_color_map=family_color_map)

    # Leyenda de familias
    legend_items = [
        mpatches.Patch(color=family_color_map[f], label=f)
        for f in families
    ]
    ax.legend(handles=legend_items, loc='upper left',
              fontsize=8, title='Familia', ncol=2)

    plt.tight_layout()
    plt.savefig(f'api_calls_total_t{int(threshold*10)}.png',
                dpi=130, bbox_inches='tight')
    plt.show()

    n_n = G_all.number_of_nodes()
    n_e = G_all.number_of_edges()
    print(f'  Umbral {threshold}: {n_n} nodos, {n_e} aristas, '
          f'densidad={nx.density(G_all):.4f}')

---
# CARACTERÍSTICA 2: Strings

Cada muestra se representa como el **conjunto de nombres de función únicos** (la parte después de `:` en cada import).  
Ejemplo: `KERNEL32.dll:LoadLibraryA` y `KERNEL32.DLL:LoadLibraryA` → ambos aportan el string `LoadLibraryA`.

Esta característica es **más permisiva** que API Calls: dos muestras que importan la misma función de distintas DLLs aún se consideran similares.

## 7a. Strings — Grafo por Familia

In [ ]:
for threshold in THRESHOLDS:
    ncols = min(n_fam, 5)
    nrows = (n_fam + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(5.5 * ncols, 5 * nrows))
    axes_flat = np.array(axes).flatten() if n_fam > 1 else [axes]

    fig.suptitle(
        f'Strings — Grafo por Familia  |  Umbral = {threshold}  (α = 0.5)',
        fontsize=13, fontweight='bold', y=1.01
    )

    for k, family in enumerate(families):
        sub   = df[df['family'] == family]
        idxs  = list(sub.index)
        sets  = list(sub['string_set'])
        color = FAMILY_COLORS[k % len(FAMILY_COLORS)]

        G = build_similarity_graph(idxs, sets, threshold, alpha=0.5)
        draw_graph(G, df, title=f'Familia: {family}',
                   node_color=color, ax=axes_flat[k])

    for ax in axes_flat[n_fam:]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'strings_familia_t{int(threshold*10)}.png',
                dpi=130, bbox_inches='tight')
    plt.show()
    print(f'  Umbral {threshold} completado.')

## 7b. Strings — Grafo de Todo el Conjunto

In [ ]:
for threshold in THRESHOLDS:
    all_idxs = list(df.index)
    all_sets = list(df['string_set'])

    G_all = build_similarity_graph(all_idxs, all_sets, threshold, alpha=0.5)

    fig, ax = plt.subplots(figsize=(14, 10))

    draw_graph(G_all, df,
               title=f'Strings — Todo el Conjunto  |  Umbral = {threshold}  (α = 0.5)',
               node_color=None, ax=ax,
               color_by_family=True, family_color_map=family_color_map)

    legend_items = [
        mpatches.Patch(color=family_color_map[f], label=f)
        for f in families
    ]
    ax.legend(handles=legend_items, loc='upper left',
              fontsize=8, title='Familia', ncol=2)

    plt.tight_layout()
    plt.savefig(f'strings_total_t{int(threshold*10)}.png',
                dpi=130, bbox_inches='tight')
    plt.show()

    n_n = G_all.number_of_nodes()
    n_e = G_all.number_of_edges()
    print(f'  Umbral {threshold}: {n_n} nodos, {n_e} aristas, '
          f'densidad={nx.density(G_all):.4f}')

---
## 8. Tabla Resumen Comparativa

Métricas del grafo (nodos, aristas, densidad) para cada combinación de característica × umbral.

In [ ]:
rows = []

for feat_name, feat_col in [('API Calls', 'api_set'), ('Strings', 'string_set')]:
    for threshold in THRESHOLDS:

        # ── Grafo global ───────────────────────────────────────────────────────
        G_g = build_similarity_graph(list(df.index),
                                     list(df[feat_col]), threshold)
        rows.append({
            'Característica': feat_name, 'Umbral': threshold,
            'Alcance': 'Global (todas las familias)',
            'Familia': '—',
            'Nodos': G_g.number_of_nodes(),
            'Aristas': G_g.number_of_edges(),
            'Densidad': round(nx.density(G_g), 4),
        })

        # ── Grafo por familia ──────────────────────────────────────────────────
        for family in families:
            sub  = df[df['family'] == family]
            idxs = list(sub.index)
            sets = list(sub[feat_col])
            G_f  = build_similarity_graph(idxs, sets, threshold)
            rows.append({
                'Característica': feat_name, 'Umbral': threshold,
                'Alcance': 'Por familia',
                'Familia': family,
                'Nodos': G_f.number_of_nodes(),
                'Aristas': G_f.number_of_edges(),
                'Densidad': round(nx.density(G_f), 4),
            })

resumen = pd.DataFrame(rows)
print(resumen.to_string(index=False))
resumen

---
## 9. Análisis de Resultados

### Efecto del umbral

| Umbral | Efecto |
|--------|--------|
| **0.3** | Umbral bajo → muchas aristas, grafo denso. Cualquier similitud leve conecta dos nodos. |
| **0.5** | Umbral medio → balance entre conectividad y especificidad. |
| **0.7** | Umbral alto → solo muestras muy similares se conectan. Revela el núcleo de cada familia. |

### API Calls vs Strings

- **API Calls** (`DLL:función`): compara el import completo. Es más estricto — dos muestras que llaman `LoadLibraryA` pero desde DLLs distintas no se consideran iguales.
- **Strings** (solo nombre de función): compara únicamente el nombre de la función, ignorando la DLL. Es más permisivo — detecta similitudes funcionales aunque los imports exactos difieran.

### Densidad dentro de una familia

- **Alta densidad interna** → muestras de esa familia comparten muchas APIs/strings → alta cohesión → familia bien definida.
- **Baja densidad interna** → muestras heterogéneas dentro de la familia → posible contaminación o familia poco homogénea.
- En el grafo global, aristas *entre* familias distintas indican que esas familias comparten comportamiento.